In [49]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parents[1]

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.config import settings
from src.aggregations import territorial, person, labor,cadunico
import pandas as pd 
import numpy as np
from pathlib import Path
import importlib

importlib.reload(settings)
importlib.reload(territorial)
importlib.reload(person)
importlib.reload(labor)
importlib.reload(cadunico)

pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.max_columns", None)

In [81]:
df_cubo = pd.read_excel(settings.CUBO_PATH)

In [2]:
df_cubo = pd.read_excel(settings.DATA_PATH /'input_data' /'dados_cubo_final_v0__2026-05-28_16-48.xlsx')

In [ ]:
# Section 1
# bignumber1.csv

# Valor executado por estado x municipio abs / percentual 
df_cubo_est = df_cubo[df_cubo['tipo_ente'] == 'ESTADO']
df_cubo_mun = df_cubo[df_cubo['tipo_ente'] == 'MUNICIPIO']

# Totais executados com ceil
valor_estados = np.ceil(df_cubo_est["valor_transacao"].sum())
valor_municipios = np.ceil(df_cubo_mun["valor_transacao"].sum())

# Valores de referência
total_estados = 1_510_000_000
total_municipios = 1_490_000_000

df_execucao = pd.DataFrame({
    "Estados_DF": [valor_estados],
    "Municipios_DF": [valor_municipios],
    "perc_executado_estados": [valor_estados / total_estados],
    "perc_executado_municipios": [valor_municipios / total_municipios],
})

df_execucao

,Estados_DF,Municipios_DF,perc_executado_estados,perc_executado_municipios
0,1.450514e+09,1.395481e+09,0.960606,0.936565


In [42]:
df_execucao.to_csv('s1_bn1.csv')

In [ ]:
# executed_value_by.csv
df_states       = territorial.executed_value_n_contemplados_qty_by(df_cubo=df_cubo, by_filter='ESTADO')
df_municipality = territorial.executed_value_n_contemplados_qty_by(df_cubo=df_cubo, by_filter='MUNICIPIO')
df_uf           = territorial.executed_value_n_contemplados_qty_by(df_cubo=df_cubo, by_filter='UF')

df_states.to_csv(settings.DATA_PATH_SECTION1 / 'executed_value_by_state.csv')
df_municipality.to_csv(settings.DATA_PATH_SECTION1 / 'executed_value_by_municipality.csv')
df_uf.to_csv(settings.DATA_PATH_SECTION1 / 'executed_value_by_uf.csv')

In [ ]:
# aggregate_faixa_valor_ju_wide_by_uf.csv
df_uf_new = territorial.aggregate_faixa_valor_ju_wide_by_uf(df_cubo=df_cubo)
df_uf_new.to_csv(settings.DATA_PATH_SECTION2 / 'aggregate_faixa_valor_ju_wide_by_uf.csv')

In [ ]:
# aggregate_faixa_valor_ju_wide_by_state.csv
df_uf_new = territorial.aggregate_faixa_valor_ju_wide_by_uf(df_cubo=df_cubo, by_filter='ESTADO')
df_uf_new.to_csv(settings.DATA_PATH_SECTION2 / 'aggregate_faixa_valor_ju_wide_by_state.csv')

In [22]:
# executed_value_by_region
df_states_region       = territorial.aggregate_execution_by_region(df_cubo=df_cubo, by_filter='ESTADO')
df_municipality_region = territorial.aggregate_execution_by_region(df_cubo=df_cubo, by_filter='MUNICIPIO')
df_uf_region           = territorial.aggregate_execution_by_region(df_cubo=df_cubo, by_filter='UF')

df_states_region.to_csv(settings.DATA_PATH_SECTION1 / 'executed_value_by_region_state.csv')
df_municipality_region.to_csv(settings.DATA_PATH_SECTION1 / 'executed_value_by_region_municipality.csv')
df_uf_region.to_csv(settings.DATA_PATH_SECTION1 / 'executed_value_by_region_uf.csv')


In [25]:
# aggregate_values_by
df_municipality_agg     = territorial.aggregate_execution_summary_by_scope(df_cubo=df_cubo, scope='MUNICIPIO')
df_state_agg            = territorial.aggregate_execution_summary_by_scope(df_cubo=df_cubo, scope='ESTADO')
df_capital_agg          = territorial.aggregate_capital_interior_summary(df_cubo=df_cubo)

df_municipality_agg.to_csv(settings.DATA_PATH_SECTION1 / 'aggregate_values_by_municipality.csv')
df_capital_agg.to_csv(settings.DATA_PATH_SECTION1 / 'aggregate_values_by_capital.csv')
df_state_agg.to_csv(settings.DATA_PATH_SECTION1 / 'aggregate_values_by_state.csv')

In [9]:
# values_by_population_size
df_population_size = territorial.aggregate_execution_by_porte_with_estado(df_cubo=df_cubo)
denominador_urbano_rural = (
    df_population_size["valor_urbano_por_porte"]
    + df_population_size["valor_rural_por_porte"]
)

df_population_size["percentual_valor_urbano_por_porte"] = np.where(
    denominador_urbano_rural.ne(0),
    df_population_size["valor_urbano_por_porte"] / denominador_urbano_rural,
    np.nan
)

df_population_size["percentual_valor_rural_por_porte"] = np.where(
    denominador_urbano_rural.ne(0),
    df_population_size["valor_rural_por_porte"] / denominador_urbano_rural,
    np.nan
)
df_population_size.to_csv(settings.DATA_PATH_SECTION1 / 'values_by_population_size.csv')

In [20]:
df_population_size.to_csv(settings.DATA_PATH_SECTION1 / 'values_by_population_size.csv')

In [ ]:
# resumo_valor_por_porte_municipio.csv
df_population_size_mean = territorial.resumo_valor_por_porte_municipio(df_cubo=df_cubo)
df_population_size_mean.to_csv(settings.DATA_PATH_SECTION1 / 'population_size_mean.csv')

In [74]:
# values_by_special_territory
df_special_territory_municipality = territorial.aggregate_special_territories_by(
    df_cubo=df_cubo, 
    categories=settings.CATEGORIES_SPECIAL_TERRITORIES, 
    by_filter="MUNICIPIO"
)

df_special_territory_state = territorial.aggregate_special_territories_by(
    df_cubo=df_cubo, 
    categories=settings.CATEGORIES_SPECIAL_TERRITORIES, 
    by_filter="ESTADO"
)

df_special_territory_uf = territorial.aggregate_special_territories_by(
    df_cubo=df_cubo, 
    categories=settings.CATEGORIES_SPECIAL_TERRITORIES, 
    by_filter="UF"
)

df_special_territory_municipality.to_csv(settings.DATA_PATH_SECTION1 / 'values_by_special_territory_municipality.csv')
df_special_territory_state.to_csv(settings.DATA_PATH_SECTION1 / 'values_by_special_territory_state.csv')
df_special_territory_uf.to_csv(settings.DATA_PATH_SECTION1 / 'values_by_special_territory_uf.csv')


In [50]:
df_cubo['cod_tipo_nome'].unique()

<ArrowStringArray>
[                      'Não especial',         'Favela e Comunidade Urbana',
 'Setor com baixo patamar domiciliar',             'Agrupamento quilombola',
               'Agrupamento indígena',             'Quartel e base militar',
                                  nan,                     'Agrovila do PA',
                  'Unidade prisional',  'Convento / hospital / ILPI / IACA',
           'Alojamento / acampamento']
Length: 11, dtype: str

In [66]:
df_cubo.head(2)

,ente,tipo_ente,tipo_documento,faixa_vlr_pago,uf,nome_ente,regiao,flag_capital,porte_populacional,Sexo,Estrangeiro,NomeNaturezaOcupacao,NomeOcupacaoPrincipal,faixa_etaria,flag_cpf_mei,cnaePrincipal,naturezaJuridica,porte,cnpj_optante_mei,raca_cor_desc_description,escolaridade_description,ind_deficiencia,tipo_deficiencia_description,tipo_vinculo_description,faixa_salarial_rais,CBO_2002_RAIS,cbo_codigo,cbo_descricao,SITUACAO,cod_situacao_nome,cod_tipo_nome,pessoaCad_cadunico,familiaPBF_cadunico,fxRendaFamiliarTotal_desc_cadunico,caracDomicilio_desc_cadunico,fxRendaPerCapita_desc_cadunico,pertence_bpc,categoria_municipio_ibge,faixa_vlr_pago_ju_bbagil,situacao_renda_cadunico,cod_cnae_principal_receita_cnpj,descr_cnae_principal_receita_cnpj,naturezajuridica_agrupada_receita_cnpj,flag_cnae_cultural,tipo_vinculo_agregado_rais,escolaridade_agregado_rais,flag_cbo_cultural_rais,flag_join_rais,quantidade,valor_transacao,min_valor_transacao,max_valor_transacao,sum_populacao
0,AC_Acre_12,ESTADO,CNPJ,1 milhão a 10 milhões,AC,Acre,Norte,False,-99,NaN,NaN,NaN,NaN,NaN,False,"{'codigo': '7820500', 'descricao': 'Locação de...","{'codigo': '2062', 'descricao': 'Sociedade Emp...",01 - MicroEmpresa-ME,0.0,NaN,NaN,NaN,NaN,NaN,NÃO SE APLICA,NaN,NaN,NaN,Urbana,Área urbana de alta densidade de edificações d...,Não especial,NaN,NaN,NaN,NaN,NaN,NaN,Interior,Acima de 200 mil,NaN,7820500.0,Locação de mão-de-obra temporária,Entidades empresariais | EPP | Microempresa,CNAE NAO CULTURAL,NaN,NaN,NaN,False,1,2034002.65,2034002.65,2034002.65,880631
1,AC_Acre_12,ESTADO,CNPJ,10 a 50 mil,AC,Acre,Norte,False,-99,NaN,NaN,NaN,NaN,NaN,False,"{'codigo': '1629301', 'descricao': 'Fabricação...","{'codigo': '2135', 'descricao': 'Empresário (I...",01 - MicroEmpresa-ME,1.0,NaN,NaN,NaN,NaN,NaN,NÃO SE APLICA,NaN,NaN,NaN,Urbana,Área urbana de alta densidade de edificações d...,Não especial,NaN,NaN,NaN,NaN,NaN,NaN,Interior,De 10 a 50 mil,NaN,1629301.0,"Fabricação de artefatos diversos de madeira, e...",Entidades empresariais | EPP | Microempresa,CNAE CULTURAL,NaN,NaN,NaN,False,1,50000.00,50000.00,50000.00,880631


In [47]:
# special_territory_w_ibge_by_brazil
df_vis_territorio_brasil = territorial.generate_special_territories_brazil_view(df_cubo=df_cubo)
df_vis_territorio_brasil.to_csv(settings.DATA_PATH_SECTION1 / 'special_territory_w_ibge_by_brazil.csv')

,territorio,Valor (R$),Quantidade de contemplados,% recurso total,% de agentes contemplados,% população no território
0,Favela e Comunidade Urbana,138146185,4555,0.05,0.03,8.00
1,Agrupamento quilombola,7369210,481,0.00,0.00,0.70
2,Agrupamento indígena,5078944,254,0.00,0.00,0.83


In [46]:
df_vis_territorio_brasil['Quantidade de contemplados'].sum()

np.int64(5290)

In [52]:
# aggregate_by_local_residencia
df_interior_rm_uf = territorial.aggregate_by_local_residencia(df_cubo=df_cubo_x, visao='uf')
df_interior_rm_uf.to_csv(settings.DATA_PATH_SECTION1 / 'aggregate_by_local_residencia_uf.csv')

In [61]:
df_cubo.head(2)

,ente,tipo_ente,tipo_documento,faixa_vlr_pago,uf,nome_ente,regiao,flag_capital,porte_populacional,Sexo,Estrangeiro,NomeNaturezaOcupacao,NomeOcupacaoPrincipal,faixa_etaria,flag_cpf_mei,cnaePrincipal,naturezaJuridica,porte,cnpj_optante_mei,raca_cor_desc_description,escolaridade_description,ind_deficiencia,tipo_deficiencia_description,tipo_vinculo_description,faixa_salarial_rais,CBO_2002_RAIS,cbo_codigo,cbo_descricao,SITUACAO,cod_situacao_nome,cod_tipo_nome,pessoaCad_cadunico,familiaPBF_cadunico,fxRendaFamiliarTotal_desc_cadunico,caracDomicilio_desc_cadunico,fxRendaPerCapita_desc_cadunico,pertence_bpc,categoria_municipio_ibge,faixa_vlr_pago_ju_bbagil,situacao_renda_cadunico,cod_cnae_principal_receita_cnpj,descr_cnae_principal_receita_cnpj,naturezajuridica_agrupada_receita_cnpj,flag_cnae_cultural,tipo_vinculo_agregado_rais,escolaridade_agregado_rais,flag_cbo_cultural_rais,flag_join_rais,quantidade,valor_transacao,min_valor_transacao,max_valor_transacao,sum_populacao
0,AC_Acre_12,ESTADO,CNPJ,1 milhão a 10 milhões,AC,Acre,Norte,False,-99,NaN,NaN,NaN,NaN,NaN,False,"{'codigo': '7820500', 'descricao': 'Locação de...","{'codigo': '2062', 'descricao': 'Sociedade Emp...",01 - MicroEmpresa-ME,0.0,NaN,NaN,NaN,NaN,NaN,NÃO SE APLICA,NaN,NaN,NaN,Urbana,Área urbana de alta densidade de edificações d...,Não especial,NaN,NaN,NaN,NaN,NaN,NaN,Interior,Acima de 200 mil,NaN,7820500.0,Locação de mão-de-obra temporária,Entidades empresariais | EPP | Microempresa,CNAE NAO CULTURAL,NaN,NaN,NaN,False,1,2034002.65,2034002.65,2034002.65,880631
1,AC_Acre_12,ESTADO,CNPJ,10 a 50 mil,AC,Acre,Norte,False,-99,NaN,NaN,NaN,NaN,NaN,False,"{'codigo': '1629301', 'descricao': 'Fabricação...","{'codigo': '2135', 'descricao': 'Empresário (I...",01 - MicroEmpresa-ME,1.0,NaN,NaN,NaN,NaN,NaN,NÃO SE APLICA,NaN,NaN,NaN,Urbana,Área urbana de alta densidade de edificações d...,Não especial,NaN,NaN,NaN,NaN,NaN,NaN,Interior,De 10 a 50 mil,NaN,1629301.0,"Fabricação de artefatos diversos de madeira, e...",Entidades empresariais | EPP | Microempresa,CNAE CULTURAL,NaN,NaN,NaN,False,1,50000.00,50000.00,50000.00,880631


# Section 2

In [27]:
df_values_by_person_type_uf = territorial.aggregate_execution_by_person_type(df_cubo=df_cubo, by_filter='UF')
df_values_by_person_type_state = territorial.aggregate_execution_by_person_type(df_cubo=df_cubo, by_filter='ESTADO')
df_values_by_person_type_municipality = territorial.aggregate_execution_by_person_type(df_cubo=df_cubo, by_filter='MUNICIPIO')

df_values_by_person_type_uf.to_csv(settings.DATA_PATH_SECTION2 / 'aggregate_execution_by_person_type_uf.csv')
df_values_by_person_type_state.to_csv(settings.DATA_PATH_SECTION2 / 'aggregate_execution_by_person_type_state.csv')
df_values_by_person_type_municipality.to_csv(settings.DATA_PATH_SECTION2 / 'aggregate_execution_by_person_type_municipality.csv')

In [ ]:
df_faixa_valor = territorial.aggregate_faixa_valor_ju_by(df_cubo=df_cubo)
# df_faixa_valor.to_csv(settings.DATA_PATH_SECTION2 / 'values_range_by_brazil_v2.csv')

In [ ]:
df_faixa_valor = territorial.aggregate_faixa_valor_ju_by(df_cubo=df_cubo, by_filter='ESTADO')
df_faixa_valor.to_csv(settings.DATA_PATH_SECTION2 / 'values_range_by_state_v2.csv')

In [38]:
df_aux = pd.read_parquet(settings.DATA_PATH / 'input_data'/ 'non-public' / 'df_aux_cubo.parquet')

In [ ]:
df_box = territorial.make_boxplot_df_faixa_valor(df_aux=df_aux)

In [71]:
df_box_qtd_contemplados = df_box[df_box['metrica'] == 'quantidade_contemplados']
df_box_qtd_contemplados.to_csv(settings.DATA_PATH_SECTION2 / 'faixa_valor_box_plot_qtd_contemplados_state.csv')

In [78]:
import pandas as pd

# Filtra apenas ESTADO
df_estado = df_aux[df_aux["tipo_ente_bbagil"] == "ESTADO"].copy()

# Garante que a coluna de valor está numérica
df_estado["valor_transacao_total_bbagil"] = pd.to_numeric(
    df_estado["valor_transacao_total_bbagil"],
    errors="coerce"
)

# Remove valores nulos
serie_valor = df_estado["valor_transacao_total_bbagil"].dropna()

# Tabela geral de percentis e quartis
df_percentis_estado = pd.DataFrame({
    "tipo_ente": ["ESTADO"],
    "quantidade_contemplados": [serie_valor.count()],
    "valor_minimo": [serie_valor.min()],
    "p1": [serie_valor.quantile(0.01)],
    "q1": [serie_valor.quantile(0.25)],
    "q2_mediana": [serie_valor.quantile(0.50)],
    "q3": [serie_valor.quantile(0.75)],
    "p99": [serie_valor.quantile(0.99)],
    "valor_maximo": [serie_valor.max()],
    "media": [serie_valor.mean()],
    "desvio_padrao": [serie_valor.std()]
})

df_percentis_estado

,tipo_ente,quantidade_contemplados,valor_minimo,p1,q1,q2_mediana,q3,p99,valor_maximo,media,desvio_padrao
0,ESTADO,22050,429.37,2824.0,12500.0,30000.0,60000.0,500000.0,22109764.92,65782.962635,264704.278241


In [ ]:
df_box_qtd_contemplados = df_box[df_box['metrica'] == 'valor_t1ransacao_total_bbagil']
df_box_qtd_contemplados.to_csv(settings.DATA_PATH_SECTION2 / 'faixa_valor_box_plot_valor_transacao_total_bbagil_state.csv')

In [54]:
importlib.reload(territorial)

<module 'src.aggregations.territorial' from 'c:\\Users\\gabiru\\Documents\\GitHub\\pnab-data-vis\\src\\aggregations\\territorial.py'>

In [39]:
# resumo_faixa_valor_por_porte.csv
df_resumo_faixas_porte = territorial.resumo_faixa_valor_por_porte(df_cubo=df_cubo)
df_resumo_faixas_porte.to_csv(settings.DATA_PATH_SECTION2 / 'faixa_valor_porte_populacional.csv')

In [40]:
df_resumo_faixas_porte

,porte_populacional,total_qtd_contemplados,total_valor_transacao,qtd_contemplados_acima_de_200_mil,perc_qtd_contemplados_acima_de_200_mil,valor_transacao_acima_de_200_mil,perc_valor_transacao_acima_de_200_mil,qtd_contemplados_de_10_a_50_mil,perc_qtd_contemplados_de_10_a_50_mil,valor_transacao_de_10_a_50_mil,perc_valor_transacao_de_10_a_50_mil,qtd_contemplados_de_50_a_200_mil,perc_qtd_contemplados_de_50_a_200_mil,valor_transacao_de_50_a_200_mil,perc_valor_transacao_de_50_a_200_mil,qtd_contemplados_de_2_a_10_mil,perc_qtd_contemplados_de_2_a_10_mil,valor_transacao_de_2_a_10_mil,perc_valor_transacao_de_2_a_10_mil,qtd_contemplados_ate_2_mil,perc_qtd_contemplados_ate_2_mil,valor_transacao_ate_2_mil,perc_valor_transacao_ate_2_mil
0,-99,22050,"1,450,514,326.10",1104,0.05,"592,272,160.59",0.41,11454,0.52,"317,106,059.64",0.22,4924,0.22,"509,229,282.04",0.35,4479,0.20,"31,762,705.32",0.02,89,0.00,"144,118.51",0.00
1,1_pequeno_i,57289,"266,445,332.42",7,0.00,"1,796,825.73",0.01,4511,0.08,"90,736,359.52",0.34,447,0.01,"33,425,238.51",0.13,23629,0.41,"106,275,294.55",0.40,28695,0.50,"34,211,614.11",0.13
2,2_pequeno_ii,38944,"234,896,275.23",22,0.00,"5,130,439.15",0.02,4424,0.11,"82,205,026.90",0.35,335,0.01,"28,685,142.34",0.12,19876,0.51,"100,181,929.29",0.43,14287,0.37,"18,693,737.55",0.08
3,3_medio,17348,"161,197,608.53",13,0.00,"4,263,457.18",0.03,3913,0.23,"77,580,099.05",0.48,254,0.01,"20,626,727.08",0.13,9523,0.55,"53,553,579.58",0.33,3645,0.21,"5,173,745.64",0.03
4,4_grande,31255,"732,942,051.71",201,0.01,"114,515,571.73",0.16,13359,0.43,"328,691,074.04",0.45,2388,0.08,"206,663,615.15",0.28,12986,0.42,"80,044,107.45",0.11,2321,0.07,"3,027,683.34",0.00


In [41]:
df_resumo_faixas_porte_cut = df_resumo_faixas_porte[df_resumo_faixas_porte['porte_populacional'] != '-99']

In [ ]:
df_territorios_especiais = territorial.resumo_territorios_especiais_por_uf(df_cubo=df_cubo)
df_territorios_especiais.to_csv(settings.DATA_PATH_SECTION2 / 'territorios_especiais_por_uf.csv')

In [57]:
df_territorios_especiais[['uf','valor_transacao_territorios_especiais']]

,uf,valor_transacao_territorios_especiais
0,AC,"2,619,104.73"
1,AL,"2,044,374.80"
2,AM,"17,170,356.45"
3,AP,"6,611,308.84"
4,BA,"14,084,958.01"
5,CE,"6,808,534.41"
6,DF,"216,015.04"
7,ES,"6,544,501.86"
8,GO,"1,097,976.11"
9,MA,"8,220,201.79"


In [ ]:
411762864.83/732942051.71

0.56179457007458

# Section 3

In [16]:
importlib.reload(person)

<module 'src.aggregations.person' from 'c:\\Users\\gabiru\\Documents\\GitHub\\pnab-data-vis\\src\\aggregations\\person.py'>

In [7]:
# aggregate_contemplados_pf_pj_proportion.csv
df_person = person.aggregate_contemplados_pf_pj_proportion(df_cubo=df_cubo)
df_person.to_csv(settings.DATA_PATH_SECTION3 / 'aggregate_contemplados_pf_pj_proportion.csv')

In [11]:
df_cubo['tipo_documento'].value_counts(dropna=False)

tipo_documento
CPF     130235
CNPJ     25363
Name: count, dtype: int64

In [8]:
# aggregate_contemplados_by_sexo_proportion.csv
df_sexo = person.aggregate_contemplados_by_sexo_proportion(df_cubo=df_cubo)
df_sexo.to_csv(settings.DATA_PATH_SECTION3 / 'aggregate_contemplados_by_sexo_proportion.csv')

In [50]:
df_sexo.head(2)

,quantidade_contemplados,perc_quantidade_contemplados,valor_contemplados,perc_valor_contemplados,quantidade_contemplados_feminino,perc_quantidade_contemplados_feminino,valor_contemplados_feminino,perc_valor_contemplados_feminino,quantidade_contemplados_masculino,perc_quantidade_contemplados_masculino,valor_contemplados_masculino,perc_valor_contemplados_masculino
0,134593,1,1254423238,1,62943,0.467654,578195818,0.460926,71650,0.532346,676227421,0.539074


In [13]:
# aggregate_valor_quantity_by_age_group_sexo_wide
df_age_group = person.aggregate_valor_quantity_by_age_group_sexo_wide(df_cubo=df_cubo)
df_age_group.to_csv(settings.DATA_PATH_SECTION3 / 'aggregate_valor_quantity_by_age_group_sexo_wide.csv')

In [ ]:
# aggregate_value_quantity_by_age_group_region_wide
df_age_region = person.aggregate_value_quantity_by_age_group_region_wide(df_cubo=df_cubo)
df_age_region.to_csv(settings.DATA_PATH_SECTION3 / 'aggregate_value_quantity_by_age_group_region_wide.csv')

# Section 4

In [43]:
importlib.reload(labor)

<module 'src.aggregations.labor' from 'c:\\Users\\gabiru\\Documents\\GitHub\\pnab-data-vis\\src\\aggregations\\labor.py'>

In [41]:
# aggregate_vinculo_formal_labor.csv
df_not_in_mercado = labor.aggregate_vinculo_formal_labor(df_cubo=df_cubo)
df_not_in_mercado.to_csv(settings.DATA_PATH_SECTION4 / 'aggregate_vinculo_formal_labor.csv')

In [42]:
# aggregate_vinculo_formal_labor_by_uf.csv
df_not_in_mercado_by_uf = labor.aggregate_vinculo_formal_labor_by_uf(df_cubo=df_cubo)
df_not_in_mercado_by_uf.to_csv(settings.DATA_PATH_SECTION4 / 'aggregate_vinculo_formal_labor_by_uf.csv')

In [43]:
# aggregate_vinculo_formal_labor_by_region.csv
df_not_in_mercado_by_region = labor.aggregate_vinculo_formal_labor_by_region(df_cubo=df_cubo)
df_not_in_mercado_by_region.to_csv(settings.DATA_PATH_SECTION4 / 'aggregate_vinculo_formal_labor_by_region.csv')


In [46]:
# aggregate_vinculo_formal_labor_by_sexo.csv
df_not_in_mercado_by_sexo = labor.aggregate_vinculo_formal_labor_by_sexo(df_cubo=df_cubo)
df_not_in_mercado_by_sexo.to_csv(settings.DATA_PATH_SECTION4 / 'aggregate_vinculo_formal_labor_by_sexo.csv')

In [50]:
# aggregate_vinculo_formal_labor_by_age_group.csv
df_not_in_mercado_by_age_group = labor.aggregate_vinculo_formal_labor_by_age_group(df_cubo=df_cubo)
df_not_in_mercado_by_age_group.to_csv(settings.DATA_PATH_SECTION4 / 'aggregate_vinculo_formal_labor_by_age_group.csv')

In [56]:
# aggregate_vinculo_formal_labor_by_raca_cor.csv
df_not_in_mercado_by_raca_cor = labor.aggregate_vinculo_formal_labor_by_raca_cor(df_cubo=df_cubo)
df_not_in_mercado_by_raca_cor.to_csv(settings.DATA_PATH_SECTION4 / 'aggregate_vinculo_formal_labor_by_raca_cor.csv')


In [ ]:
# aggregate_raca_cor_vinculo_formal_labor_by_sexo
df_not_in_mercado_by_raca_cor_sexo = labor.aggregate_raca_cor_vinculo_formal_labor_by_sexo(df_cubo=df_cubo)
df_not_in_mercado_by_raca_cor_sexo.to_csv(settings.DATA_PATH_SECTION4 / 'aggregate_raca_cor_vinculo_formal_labor_by_sexo.csv')

In [67]:
# aggregate_vinculo_formal_labor_by_escolaridade.csv
df_not_in_mercado_escolaridade = labor.aggregate_vinculo_formal_labor_by_escolaridade(df_cubo=df_cubo)
df_not_in_mercado_escolaridade.to_csv(settings.DATA_PATH_SECTION4 / 'aggregate_vinculo_formal_labor_by_escolaridade.csv')

In [69]:
# aggregate_vinculo_trabalho_formal_by_escolaridade_clean.csv
df_not_in_mercado_escolaridade_clean = labor.aggregate_vinculo_trabalho_formal_by_escolaridade_sem_sem_informacao(df_cubo=df_cubo)
df_not_in_mercado_escolaridade_clean.to_csv(settings.DATA_PATH_SECTION4 / 'aggregate_vinculo_trabalho_formal_by_escolaridade_clean.csv')

In [ ]:
# CBOS

np.float64(12415168.75)

In [46]:
# aggregate_cbo_rais.csv
df_cbo_rais = labor.aggregate_cbo_rais(df_cubo=df_cubo)
df_cbo_rais.to_csv(settings.DATA_PATH_SECTION4 / 'aggregate_cbo_rais.csv')

In [1]:
df_cubo

NameError: name 'df_cubo' is not defined

# Section 5

In [177]:
importlib.reload(cadunico)

<module 'src.aggregations.cadunico' from 'c:\\Users\\gabiru\\Documents\\GitHub\\pnab-data-vis\\src\\aggregations\\cadunico.py'>

In [47]:
df_cad_unico = pd.read_parquet(settings.DATA_PATH / 'input_data' / 'non-public' /'dim_cadunico__2026-05-19_18-17.parquet')

In [100]:
# aggregate_cadunico_summary.csv
df_cubo_cadunico = cadunico.aggregate_cadunico_summary(df_cubo=df_cubo)
df_cubo_cadunico.to_csv(settings.DATA_PATH_SECTION5 / 'aggregate_cadunico_summary.csv')

In [101]:
# aggregate_cadunico_profile_summary.csv
df_cubo_cadunico_sexo_idade = cadunico.aggregate_cadunico_profile_summary(df_cubo=df_cubo)
sexo = df_cubo_cadunico_sexo_idade[df_cubo_cadunico_sexo_idade['dimensao'] == 'Sexo']
faixa_etaria = df_cubo_cadunico_sexo_idade[df_cubo_cadunico_sexo_idade['dimensao'] == 'Faixa etária']
sexo.to_csv(settings.DATA_PATH_SECTION5 / 'aggregate_cadunico_profile_summary_by_sexo.csv')
faixa_etaria.to_csv(settings.DATA_PATH_SECTION5 / 'aggregate_cadunico_profile_summary_by_faixa_etaria.csv')


In [ ]:
# aggregate_cadunico_faixa_etaria_by_sexo.csv
df_cubo_cadunico_sexo_idade_juntos = cadunico.aggregate_cadunico_faixa_etaria_by_sexo(df_cubo=df_cubo)
df_cubo_cadunico_sexo_idade_juntos.to_csv(settings.DATA_PATH_SECTION5 / 'aggregate_cadunico_faixa_etaria_by_sexo.csv')

In [ ]:
# aggregate_cadunico_by_situacao_renda.csv
df_cad_unico_situacao_renda = cadunico.aggregate_cadunico_by_situacao_renda(df_cubo=df_cubo)
df_cad_unico_situacao_renda.to_csv(settings.DATA_PATH_SECTION5 / 'aggregate_cadunico_by_situacao_renda.csv')


In [ ]:
# aggregate_cadunico_by_fx_renda_per_capita.csv
df_cad_unico_situacao_faixa_renda_percapita = cadunico.aggregate_cadunico_by_fx_renda_per_capita(df_cubo=df_cubo)
df_cad_unico_situacao_faixa_renda_percapita.to_csv(settings.DATA_PATH_SECTION5 / 'aggregate_cadunico_by_fx_renda_per_capita.csv')

In [ ]:
# aggregate_cadunico_by_situacao_domicilio.csv
df_cad_unico_domicilio_situacao = cadunico.aggregate_cadunico_by_situacao_domicilio(df_cubo=df_cubo)
df_cad_unico_domicilio_situacao.to_csv(settings.DATA_PATH_SECTION5 / 'aggregate_cadunico_by_situacao_domicilio.csv')

In [187]:
# aggregate_cadunico_by_population_size.csv
df_cad_unico_porte_populacional = cadunico.aggregate_cadunico_by_population_size(df_cubo=df_cubo)
df_cad_unico_porte_populacional.to_csv(settings.DATA_PATH_SECTION5 / 'aggregate_cadunico_by_population_size.csv')


In [188]:
# aggregate_cadunico_by_uf.csv
df_cad_unico_by_uf = cadunico.aggregate_cadunico_by_uf(df_cubo=df_cubo)
df_cad_unico_by_uf.to_csv(settings.DATA_PATH_SECTION5 / 'aggregate_cadunico_by_uf.csv')

In [172]:
# aggregate_cadunico_by_value_group.csv
df_cad_unic_faixa_valor = cadunico.aggregate_cadunico_by_value_group(df_cubo=df_cubo)
df_cad_unic_faixa_valor.to_csv(settings.DATA_PATH_SECTION5 / 'aggregate_cadunico_by_value_group.csv')


In [173]:
# aggregate_bolsa_familia_summary.csv
df_cad_unico_bpf = cadunico.aggregate_bolsa_familia_summary(df_cubo=df_cubo)
df_cad_unico_bpf.to_csv(settings.DATA_PATH_SECTION5 / 'aggregate_bolsa_familia_summary.csv')

In [180]:
# aggregate_bpc_summary.csv
df_cad_unico_bpc = cadunico.aggregate_bpc_summary(df_cubo=df_cubo)
df_cad_unico_bpc.to_csv(settings.DATA_PATH_SECTION5 / 'aggregate_bpc_summary.csv')

In [181]:
df_cpf_receita = pd.read_parquet(settings.DATA_PATH / 'input_data' / 'non-public' /'dim_cpf_receita__2026_05-19-13_09.parquet')

In [183]:
df_cpf_receita[df_cpf_receita['sexo_receita_cpf'] == 'Feminino']['cpf_receita_cpf'].nunique()

61026

In [184]:
df_cpf_receita[df_cpf_receita['sexo_receita_cpf'] == 'Masculino']['cpf_receita_cpf'].nunique()

68615

In [186]:
61026/(61026+68615)

0.47073071019199175